# Omni ChromHMM Analysis

Analysis, cross-segmentation comparison and inter-dataset summary plots.

Run the Snakemake pipeline first to produce the segmentations (`{ds}/.done`); then run
this notebook top-to-bottom:
1. **Per-segmentation analysis** — `analyze.run_analyze` / `analyze_peaks.run_analyze_peaks`
2. **Cross-segmentation comparison** — `compare.run_compare` / `compare_methods.run_compare_methods`
3. **Inter-dataset comparison & summary plots** — `compare`, `compare_out`, `summary_plots`, `emission_similarity`
4. **Results** — every plot displayed inline, grouped into five sections:
   (1) peaks number and lengths, (2) segmentation states number,
   (3) segmentation states lengths, (4) segmentation compositions,
   and (5) all other analyses.

In [ ]:
import os
import sys
import glob
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, HTML
import seaborn as sns
import pickle
from collections import defaultdict

# Load configuration
config_path = os.path.abspath(os.path.expanduser("~/work/omni-chromhmm/config.yaml"))
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(config_path)
scripts_dir = os.path.join(project_root, "scripts", "analysis")
scripts_rules_dir = os.path.join(project_root, "scripts", "rules")
workdir = os.path.expanduser(config.get("workdir", "."))

# Import the analysis methods directly (no CLI / subprocess).
sys.path.insert(0, scripts_dir)
sys.path.insert(0, scripts_rules_dir)

import importlib
import analyze
import analyze_peaks
import compare
import compare_methods
import compare_inter_dataset as compare_out
import emission_similarity
import match
import summary_plots
from concurrent.futures import ProcessPoolExecutor, as_completed

# Re-import the analysis modules so edits to scripts/analysis/*.py are picked up when this
# cell is re-run, without needing a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, compare_methods,
           compare_out, emission_similarity, match, summary_plots):
    importlib.reload(_m)

# Run everything relative to the pipeline working directory.
os.chdir(workdir)
print(f"Project root: {project_root}")
print(f"Scripts dir : {scripts_dir}")
print(f"Working dir : {workdir}")

# --- Parameters (mirrors the Snakefile) ----------------------------------
P = config["params"]
TOOLS = config["tools"]
DATASETS = config["datasets"]
MARKS = P["marks"]
CHROMHMM_BIN = P["chromhmm_bin"]
OMNI_BIN = P["omni_bin"]
HOMER_BIN = P["homer_bin"]
MACS2_BIN = P["macs2_bin"]
NSTATES = P["n_states"]
MATCH_METHOD = "matched"
CALLER_BIN = {"omni": OMNI_BIN, "homer": HOMER_BIN, "macs2": MACS2_BIN}

DO_REPLICATES = P.get("replicates", False)

# Peak callers to include. Edit to match the segmentations you actually produced;
# missing files are skipped gracefully throughout the notebook.
CALLERS = ["homer", "macs2", "omni"]

COORDS_DIR = os.path.join(workdir, TOOLS["coords_dir"])
GENCODE_GTF = os.path.join(workdir, TOOLS["gencode_gtf"])
MARKUPS_DIR = os.path.join(project_root, "markups")

# De-novo methods compared across datasets
INTER_DS_METHODS = (
        ["chromhmm_default"]
        + [f"kmeans_{c}" for c in CALLERS]
)
CHIP_DATASETS = [d for d in DATASETS if not d.endswith("_mint")]
MINT_DATASETS = [d for d in DATASETS if d.endswith("_mint")]
REP_DATASETS = [d for d in DATASETS if DO_REPLICATES and DATASETS[d].get("replicates")]


# --- Path helpers (mirror the Snakefile functions) -----------------------
def ds_of(folder):
    return folder.split("/")[0]


def folders_of(ds):
    fl = [ds]
    if DO_REPLICATES and DATASETS[ds].get("replicates"):
        fl += [f"{ds}/rep1", f"{ds}/rep2"]
    return fl


def ref_bed_path(ds):
    return f"{ds}/{DATASETS[ds]['ref_chromhmm']}_chromhmm.bed"


def seg_bin(path):
    for caller, size in CALLER_BIN.items():
        if f"/{caller}/" in path:
            return size
    return CHROMHMM_BIN


def inter_ds_bed(ds, method):
    cell = DATASETS[ds]["cell"]
    sfx = MATCH_METHOD
    if method == "chromhmm_default":
        return f"{ds}/chromhmm_default_result/{cell}_{NSTATES}_dense_{sfx}.bed"
    model, caller = method.split("_")  # kmeans , omni|homer|macs2
    return f"{ds}/{caller}/{caller}_kmeans_states_{sfx}.bed"


def existing(paths):
    """Keep only paths that exist on disk (skip segmentations not produced)."""
    return [p for p in paths if os.path.exists(p)]


print(f"Callers       : {CALLERS}")
print(f"Match variant : {MATCH_METHOD}")
print(f"Inter methods : {INTER_DS_METHODS}")

# --- Metadata for plots and display --------------------------------------
VARIANT = MATCH_METHOD  # default match variant, e.g. "comb"
SP = "out/summary_plots"  # cross-dataset summary plots
REF = "out/reference"  # ENCODE reference plots

DS_TITLE = {
    "imr90": "IMR90 (ChIP-seq)",
    "monocytes": "Monocytes (ChIP-seq)",
    "monocytes_mint": "Monocytes (Mint-ChIP)",
    "gm12878_mint": "GM12878 (Mint-ChIP)",
    "spleen": "Spleen (ChIP-seq)",
}

# Per-method segmentations shown in per-dataset grids (de-novo + reference).
METHOD_LABELS = [
    ("ref", "ENCODE reference"),
    ("chromhmm_default", "Default ChromHMM"),
    ("kmeans_omni", "KMeans OmniPeak"),
    ("kmeans_homer", "KMeans HOMER"),
    ("kmeans_macs2", "KMeans MACS2"),
]


# 1. Performing computations


## Run Analysis
Executing the analysis for all datasets and folders.


In [ ]:
annotations = sorted(glob.glob(os.path.join(COORDS_DIR, "*.bed.gz")))
if not annotations:
    print(f"WARNING: No standard annotations found in {COORDS_DIR}!")
    # Try relative path as fallback
    annotations = sorted(glob.glob(os.path.join(TOOLS["coords_dir"], "*.bed.gz")))
    if annotations:
        print(f"Found {len(annotations)} annotations using relative path.")

futures = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    for ds, cfg in DATASETS.items():
        cell = cfg["cell"]
        folders = [ds]
        if DO_REPLICATES and cfg.get("replicates"):
            folders += [f"{ds}/rep1", f"{ds}/rep2"]

        # Peak analysis
        peaks_outdir = f"{ds}/peaks"
        if not os.path.exists(os.path.join(peaks_outdir, "peak_stats.tsv")):
            print(f"Analyzing peaks for {ds}...")
            futures.append(
                executor.submit(
                    analyze_peaks.run_analyze_peaks,
                    ds=ds, cell=cell, marks=list(MARKS), outdir=peaks_outdir,
                    omni_bin=P["omni_bin"], chromhmm_bin=CHROMHMM_BIN,
                )
            )

        # RNA-seq / ATAC extra annotations if available
        ds_annotations = list(annotations)
        if cfg.get("atac"):
            ds_annotations.append(f"{ds}/atac_{cfg['atac']}.bed.gz")

        # Segmentations analysis
        for folder in folders:
            # Reference
            ref_bed = f"{ds}/{cfg['ref_chromhmm']}_chromhmm.bed"
            ref_outdir = f"out/{folder}/ref"
            if not os.path.exists(os.path.join(ref_outdir, "report.tsv")):
                print(f"Analyzing reference segmentations in {folder}...")
                futures.append(
                    executor.submit(
                        analyze.run_analyze,
                        seg=ref_bed, bin_size=CHROMHMM_BIN,
                        outdir=ref_outdir,
                        inputs=None,
                        annotations=ds_annotations,
                        bw_emissions=ref_bed.replace(".bed", ".bw_emissions.npz"),
                        rnaseq=None,
                        gtf=None,
                        emissions_only=False,
                    )
                )

            # Default ChromHMM
            default_seg = f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense.bed"
            default_outdir = f"out/{folder}/chromhmm_default_dense"
            if not os.path.exists(os.path.join(default_outdir, "report.tsv")):
                print(f"Analyzing default ChromHMM in {folder}...")
                futures.append(
                    executor.submit(
                        analyze.run_analyze,
                        seg=default_seg, bin_size=CHROMHMM_BIN,
                        outdir=default_outdir,
                        inputs=[f"{folder}/chromhmm_default/*.txt"],
                        annotations=None,
                        bw_emissions=default_seg.replace(".bed", ".bw_emissions.npz"),
                        rnaseq=None,
                        gtf=None,
                        emissions_only=False,
                    )
                )

            variant = "matched"
            matched_seg = f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense_matched.bed"
            matched_outdir = f"out/{folder}/matched/chromhmm_default"
            if not os.path.exists(os.path.join(matched_outdir, "report.tsv")):
                print(f"Analyzing matched ChromHMM in {folder}...")
                futures.append(
                    executor.submit(
                        analyze.run_analyze,
                        seg=matched_seg, bin_size=CHROMHMM_BIN,
                        outdir=matched_outdir,
                        inputs=[f"{folder}/chromhmm_default/*.txt"],
                        annotations=ds_annotations,
                        bw_emissions=matched_seg.replace(".bed", ".bw_emissions.npz"),
                        rnaseq=f"{ds}/rnaseq_{cfg['rnaseq']}.tsv" if cfg.get("rnaseq") else None,
                        gtf=GENCODE_GTF if cfg.get("rnaseq") else None,
                        emissions_only=False,
                    )
                )

            for caller in CALLERS:
                cbin = CALLER_BIN[caller]
                peaks_dir = f"{folder}/{caller}/chromhmm_peaks"

                # KMeans
                kmeans_seg = f"{folder}/{caller}/{caller}_kmeans_states_matched.bed"
                kmeans_outdir = f"out/{folder}/matched/kmeans_{caller}"
                if not os.path.exists(os.path.join(kmeans_outdir, "report.tsv")):
                    print(f"Analyzing KMeans {caller} in {folder}...")
                    futures.append(
                        executor.submit(
                            analyze.run_analyze,
                            seg=kmeans_seg, bin_size=cbin,
                            outdir=kmeans_outdir,
                            inputs=[f"{peaks_dir}/*.txt.gz"],
                            annotations=ds_annotations,
                            bw_emissions=kmeans_seg.replace(".bed", ".bw_emissions.npz"),
                            rnaseq=f"{ds}/rnaseq_{cfg['rnaseq']}.tsv" if cfg.get("rnaseq") else None,
                            gtf=GENCODE_GTF if cfg.get("rnaseq") else None
                        )
                    )

for fut in as_completed(futures):
    try:
        fut.result()
    except Exception as e:
        print(f"  ERROR: {e}")

print('Done')

## Cross-segmentation comparison

Per dataset: transition-matrix entropy, pairwise Cohen's
κ and Jaccard similarity, emission similarity and segment-length statistics, plus the
unified method comparison table. Requires the per-segmentation analysis above to have
run (it reads out/<dataset>/<variant>/.../jaccard.tsv and the .bin_emissions.npz files).


In [ ]:
# Cross-segmentation comparison per dataset
def compare_beds_for_folder(folder):
    cell = DATASETS[ds_of(folder)]["cell"]
    beds = [f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense_matched.bed"]
    for caller in CALLERS:
        beds.append(f"{folder}/{caller}/{caller}_kmeans_states_matched.bed")
    return beds


def ds_compare_segs(ds):
    segs = [ref_bed_path(ds)]
    for folder in folders_of(ds):
        segs += compare_beds_for_folder(folder)
    return segs


variant = MATCH_METHOD
dataset_args = []
for ds in DATASETS:
    segs = existing(ds_compare_segs(ds))
    if len(segs) < 2:
        print(f"{ds}: found {len(segs)} segmentation(s) for '{variant}', skipping comparison")
        continue
    bins = [seg_bin(p) for p in segs]
    dataset_args.append((ds, segs, bins))

print(f"Comparing {len(dataset_args)} datasets (variant={variant}) ...")

# 1. Atomic run_compare (cross-segmentation similarity metrics & stats)
# Note: run_compare internally parallelizes pair comparisons.
for ds, segs, bins in dataset_args:
    comp_outdir = f"out/{ds}/{variant}"
    if os.path.exists(os.path.join(comp_outdir, "comparison_all_pairs.tsv")):
        print(f"  {ds} comparison already present, skipping.")
        continue
    print(f"  {ds} ...")
    try:
        compare.run_compare(seg=segs, bins=bins,
                            outdir=comp_outdir,
                            analysis_dir=f"out/{ds}/{variant}")
    except Exception as e:
        print(f"  ERROR comparing {ds}: {e}")

# 2. Atomic run_compare_methods (aggregate results & per-method plots)
for ds, segs, bins in dataset_args:
    methods_outdir = f"out/{ds}/{variant}"
    if os.path.exists(os.path.join(methods_outdir, "comparison_table.tsv")):
        print(f"  {ds} compare methods already present, skipping.")
        continue
    print(f"  {ds} compare methods ...")
    try:
        compare_methods.run_compare_methods(
                        analysis_dir=f"out/{ds}/{variant}",
                        comparison_dir=f"out/{ds}/{variant}",
                        outdir=methods_outdir,
                        ref_dir=f"out/{ds}")
    except Exception as e:
        print(f"  ERROR compare_methods for {ds}: {e}")


## Inter-dataset comparison


In [ ]:
# Inter-dataset comparison
ds_list = list(DATASETS)
cells = [DATASETS[d]["cell"] for d in ds_list]
sp_out = "out/summary_plots"
os.makedirs(sp_out, exist_ok=True)


def _try(label, fn):
    """Run one plotting step; report and continue on failure (e.g. missing inputs)."""
    try:
        fn()
    except Exception as e:
        print(f"  SKIP {label}: {e}")


# 1. Per-method cross-dataset comparison (every dataset pair).
for method in INTER_DS_METHODS:
    pairs = [(d, inter_ds_bed(d, method)) for d in ds_list]
    pairs = [(d, p) for d, p in pairs if os.path.exists(p)]
    if len(pairs) < 2:
        print(f"  SKIP inter compare {method}: <2 datasets with this segmentation")
        continue

    outdir = f"out/{method}"
    if os.path.exists(os.path.join(outdir, "comparison_all_pairs.tsv")):
        print(f"  {method} inter-dataset comparison already present, skipping.")
        continue

    segs = [p for _, p in pairs]
    labels = [f"{d}:{method}" for d, _ in pairs]
    bins = [seg_bin(p) for p in segs]
    print(f"Inter-dataset compare: {method} ({len(segs)} datasets)")
    _try(f"compare {method}",
         lambda segs=segs, bins=bins, labels=labels, method=method, outdir=outdir:
         compare.run_compare(seg=segs, bins=bins, labels=labels, all_pairs=True,
                             outdir=outdir))

# 2. Aggregate per-method kappa matrices into one cross-dataset table.
if not os.path.exists("out/comparison_table.tsv"):
    _try("comparison_table", lambda: compare_out.run_compare_out(
        methods=INTER_DS_METHODS, indir="out",
        outfile="out/comparison_table.tsv"))


In [ ]:
# Pairwise similarity among all ENCODE reference segmentations + reference plots.
ref_segs = sorted(glob.glob(os.path.join(MARKUPS_DIR, "15state", "*.bed.gz")),
                  key=lambda p: "_".join(os.path.basename(p).replace(".bed.gz", "").split("_")[1:]))
if not ref_segs:
    print("No reference markups in", os.path.join(MARKUPS_DIR, "15state"))
else:
    ref_outdir = "out/reference"
    if os.path.exists(os.path.join(ref_outdir, "comparison_all_pairs.tsv")):
        print("Reference comparison already present, skipping.")
    else:
        ref_labels = ["_".join(os.path.basename(p).replace(".bed.gz", "").split("_")[1:])
                      for p in ref_segs]
        _try("reference compare", lambda: compare.run_compare(
            seg=ref_segs, bins=CHROMHMM_BIN, labels=ref_labels, all_pairs=True,
            outdir=ref_outdir))


### Optimal Number of States Analysis (Computation)
1. **Elbow Method (Inertia)**: Look for the point where the rate of decrease in inertia significantly slows down.
2. **Silhouette Score**: Higher average silhouette scores indicate better-defined clusters.


In [ ]:
# Silhouette and Elbow analysis for all datasets (Computation)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import subprocess

n_states_range = range(2, 31)
sample_size = 10_000
n_states_results = {}
n_states_results_path = "out/optimal_n_states.tsv"

# Helper for per-dataset per-method caching
os.makedirs("out/n_states", exist_ok=True)

def get_cache_path(ds, method):
    return f"out/n_states/{ds}_{method}.tsv"

def load_from_cache(ds, method):
    cache_path = get_cache_path(ds, method)
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, sep="\t")
        # Check if all n_states are present and no NaNs in entropy if possible
        if set(df['n_states']) == set(n_states_range):
            df = df.sort_values("n_states")
            return {
                "inertia": df["inertia"].tolist(),
                "silhouette": df["silhouette"].tolist(),
                "entropy": df["entropy"].tolist()
            }
    return None

def save_to_cache(ds, method, res):
    cache_path = get_cache_path(ds, method)
    rows = []
    for i, n in enumerate(n_states_range):
        rows.append({
            "n_states": n,
            "inertia": res["inertia"][i],
            "silhouette": res["silhouette"][i],
            "entropy": res["entropy"][i]
        })
    pd.DataFrame(rows).to_csv(cache_path, sep="\t", index=False)

# Helper for entropy
def compute_entropy_from_labels(labels, bin_size):
    # labels is a numpy array of state indices
    segs = []
    for i, label in enumerate(labels):
        segs.append(["chr22", i * bin_size, (i + 1) * bin_size, str(label)])
    states, counts, state_bp = analyze.build_transition_matrix(segs, bin_size)
    if not states:
        return 0
    total_H, _, _, _ = analyze.transition_entropy(states, counts, state_bp)
    return total_H

if os.path.exists(n_states_results_path):
    print(f"Loading optimal n_states results from {n_states_results_path}...")
    df_results = pd.read_csv(n_states_results_path, sep="\t")
    for (ds, method), group in df_results.groupby(['dataset', 'method']):
        if set(group['n_states']) == set(n_states_range):
            group = group.sort_values("n_states")
            n_states_results[(ds, method)] = {
                "inertia": group["inertia"].tolist(),
                "silhouette": group["silhouette"].tolist(),
                "entropy": group["entropy"].tolist()
            }

for ds in DATASETS:
    cell = DATASETS[ds]["cell"]
    for caller in CALLERS:
        if (ds, caller) in n_states_results and not np.isnan(n_states_results[(ds, caller)]["entropy"]).any():
            continue

        cache_res = load_from_cache(ds, caller)
        if cache_res and not np.isnan(cache_res["entropy"]).any():
            n_states_results[(ds, caller)] = cache_res
            continue

        # Try both prefixed and non-prefixed names
        paths_to_try = [
            os.path.join(ds, caller, "chromhmm_peaks", f"{cell}_chr22_binary.txt.gz"),
            os.path.join(ds, caller, "chromhmm_peaks", f"chr22_binary.txt.gz")
        ]
        data_path = None
        for p in paths_to_try:
            if os.path.exists(p):
                data_path = p
                break
        
        if not data_path:
            continue

        print(f"Loading data for {ds} {caller} from {data_path}...")
        chrom, marks, X = analyze.load_binary(data_path)
        bin_size = P.get(f"{caller}_bin", 200)

        if X.shape[0] > sample_size:
            np.random.seed(42)
            idx = np.random.choice(X.shape[0], sample_size, replace=False)
            X_sample = X[idx]
        else:
            X_sample = X

        inertia = []
        silhouette_scores = []
        entropies = []

        base_inertia = np.sum((X - X.mean(axis=0))**2)
        if base_inertia == 0: base_inertia = 1

        for n in n_states_range:
            kmeans = KMeans(n_clusters=n, init='k-means++', random_state=42, n_init=10)
            kmeans.fit(X)
            inertia.append(kmeans.inertia_ / base_inertia)
            
            sample_labels = kmeans.predict(X_sample)
            silhouette_scores.append(silhouette_score(X_sample, sample_labels))
            entropies.append(compute_entropy_from_labels(kmeans.labels_, bin_size))

        n_states_results[(ds, caller)] = {
            "inertia": inertia,
            "silhouette": silhouette_scores,
            "entropy": entropies
        }
        save_to_cache(ds, caller, n_states_results[(ds, caller)])

    # ChromHMM LearnModel on chr22
    if (ds, "chromhmm") in n_states_results and not np.isnan(n_states_results[(ds, "chromhmm")]["entropy"]).any():
        pass
    else:
        cache_res = load_from_cache(ds, "chromhmm")
        if cache_res and not np.isnan(cache_res["entropy"]).any():
            n_states_results[(ds, "chromhmm")] = cache_res
        else:
            indir = os.path.join(ds, "chromhmm_default")
            if os.path.exists(indir):
                chr22_indir = os.path.join(ds, "chromhmm_chr22")
                os.makedirs(chr22_indir, exist_ok=True)
                src = None
                for f in os.listdir(indir):
                    if f.endswith("chr22_binary.txt.gz") or f.endswith("chr22_binary.txt"):
                        if cell in f:
                            src = os.path.join(indir, f)
                            break
                if src:
                    dst = os.path.join(chr22_indir, os.path.basename(src))
                    if not os.path.exists(dst):
                        os.symlink(os.path.abspath(src), dst)
                    
                    chromhmm_entropies = []
                    for n in n_states_range:
                        outdir = os.path.join(ds, f"chromhmm_chr22_res_{n}")
                        seg_path = os.path.join(outdir, f"{cell}_{n}_segments.bed")
                        if not os.path.exists(seg_path):
                            cmd = [
                                "java", "-mx4000M", "-jar", os.path.join(workdir, config["tools"]["chromhmm_jar"]),
                                "LearnModel", chr22_indir, outdir, str(n), config["params"]["genome"]
                            ]
                            print(f"Running ChromHMM for {ds} n={n}...")
                            subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                        
                        if os.path.exists(seg_path):
                            segs = analyze.load_bed(seg_path)
                            states, counts, state_bp = analyze.build_transition_matrix(segs, 200)
                            total_H, _, _, _ = analyze.transition_entropy(states, counts, state_bp)
                            chromhmm_entropies.append(total_H)
                        else:
                            chromhmm_entropies.append(np.nan)
                    
                    n_states_results[(ds, "chromhmm")] = {
                        "inertia": [np.nan] * len(n_states_range),
                        "silhouette": [np.nan] * len(n_states_range),
                        "entropy": chromhmm_entropies
                    }
                    save_to_cache(ds, "chromhmm", n_states_results[(ds, "chromhmm")])

if n_states_results:
    rows = []
    for (ds, method), res in n_states_results.items():
        for i, n in enumerate(n_states_range):
            rows.append({
                "dataset": ds,
                "method": method,
                "n_states": n,
                "inertia": res["inertia"][i],
                "silhouette": res["silhouette"][i],
                "entropy": res["entropy"][i]
            })
    os.makedirs("out", exist_ok=True)
    pd.DataFrame(rows).to_csv(n_states_results_path, sep="\t", index=False)


# 2. Doing plotting


In [ ]:
# 2. Doing plotting
_try("reference summary plots", lambda: summary_plots.run_summary_plots(
        markups_dir=MARKUPS_DIR,
        ref_composition_outfile="out/reference/state_composition.png",
        ref_comp_matrix="out/reference/composition_similarity_matrix.tsv",
        ref_kappa_matrix="out/reference/kappa_matrix.tsv",
        ref_jaccard_matrix="out/reference/jaccard_similarity_matrix.tsv",
        ref_dist_outfile="out/reference/similarity_distribution.png",
        ref_comp_noqh_matrix="out/reference/composition_noqh_similarity_matrix.tsv",
        ref_kappa_noqh_matrix="out/reference/kappa_noqh_matrix.tsv",
        ref_jaccard_noqh_matrix="out/reference/jaccard_noqh_matrix.tsv",
        ref_dist_noqh_outfile="out/reference/similarity_distribution_noqh.png"))


In [ ]:
# Cross-dataset summary bar / violin / distribution plots.
methods_dirs = [f"out/{d}/{MATCH_METHOD}" for d in ds_list]
analysis_dirs = [f"out/{d}/{MATCH_METHOD}" for d in ds_list]

_try("summary bar plots", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, methods_dirs=methods_dirs, analysis_dirs=analysis_dirs,
    methods=INTER_DS_METHODS, outdir=sp_out))

_try("state length violin", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    violin_outfile=f"{sp_out}/state_length_comparison.png"))

_try("state coverage", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    state_coverage_outfile=f"{sp_out}/state_coverage.png"))

_try("peak count", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_count_outfile=f"{sp_out}/peak_count.png"))
_try("peak length", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_length_outfile=f"{sp_out}/peak_length.png"))
_try("peak gap violin", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_gap_violin_outfile=f"{sp_out}/peak_gap_violin.png"))

_try("method similarity distribution", lambda: summary_plots.run_summary_plots(
    method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
    method_sim_dist_outfile=f"{sp_out}/method_similarity_distribution.png",
    method_sim_dist_noqh_outfile=f"{sp_out}/method_similarity_distribution_noqh.png"))

if CHIP_DATASETS and MINT_DATASETS:
    _try("ChIP vs Mint similarity distribution", lambda: summary_plots.run_summary_plots(
        method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
        method_sim_dist_group_a=CHIP_DATASETS, method_sim_dist_group_b=MINT_DATASETS,
        method_sim_dist_filtered_outfile=f"{sp_out}/method_similarity_distribution_chip_vs_mint.png",
        method_sim_dist_filtered_noqh_outfile=f"{sp_out}/method_similarity_distribution_chip_vs_mint_noqh.png"))

if REP_DATASETS:
    _try("replicate consistency", lambda: summary_plots.run_summary_plots(
        datasets=REP_DATASETS, methods_dirs=[f"out/{d}/{VARIANT}" for d in REP_DATASETS],
        methods=INTER_DS_METHODS, rep_consistency_outdir=sp_out))

_try("per-dataset state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, methods=INTER_DS_METHODS,
    nstates=NSTATES, match_method=MATCH_METHOD, 
    method_ds_composition_outdir=sp_out,
    all_methods_composition_outdir=sp_out))
_try("mean state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    method_composition_outfile=f"{sp_out}/method_state_composition.png"))

# --- Per-assay (ChIP-seq vs Mint-ChIP) splits of the cross-dataset summaries ---
# Regenerate the peaks, total-segments and per-dataset ENCODE-reference plots
# separately for each assay group, into chip/ and mint/ subdirs of sp_out.
for _grp, _gname, _dss in [("chip", "ChIP-seq", CHIP_DATASETS),
                           ("mint", "Mint-ChIP", MINT_DATASETS)]:
    if not _dss:
        continue
    _gdir = f"{sp_out}/{_grp}"
    os.makedirs(_gdir, exist_ok=True)
    _mdirs = [f"out/{d}/{MATCH_METHOD}" for d in _dss]
    _adirs = [f"out/{d}/{MATCH_METHOD}" for d in _dss]
    _try(f"summary bars [{_grp}]", lambda dss=_dss, md=_mdirs, ad=_adirs, gd=_gdir:
    summary_plots.run_summary_plots(datasets=dss, methods_dirs=md, analysis_dirs=ad,
                                    methods=INTER_DS_METHODS, outdir=gd,
                                    all_methods_composition_outdir=gd))
    _try(f"peaks [{_grp}]", lambda dss=_dss, gd=_gdir:
    summary_plots.run_summary_plots(datasets=dss, workdir=workdir, methods=INTER_DS_METHODS,
                                    peak_count_outfile=f"{gd}/peak_count.png",
                                    peak_length_outfile=f"{gd}/peak_length.png",
                                    peak_gap_violin_outfile=f"{gd}/peak_gap_violin.png"))
    _try(f"reference n_segments [{_grp}]", lambda dss=_dss, md=_mdirs, gd=_gdir, gn=_gname:
    summary_plots.plot_reference_n_segments(
        dss, md, [DATASETS[d]["cell"] for d in dss],
        f"{gd}/reference_n_segments.png", f"ENCODE reference segments — {gn}"))


In [ ]:
# Emission discriminability (Gini) per dataset + summary.
analysis_dirs_matched = [f"out/{d}/matched" for d in ds_list]
if not os.path.exists(f"{sp_out}/emission_gini_summary.png"):
    _try("emission similarity", lambda: emission_similarity.run_emission_similarity(
        datasets=ds_list, analysis_dirs=analysis_dirs_matched, methods=INTER_DS_METHODS,
        outdir=sp_out))
if not os.path.exists(f"{sp_out}/out_binem_similarity.png"):
    _try("binarized emission similarity", lambda: emission_similarity.run_emission_similarity(
        datasets=ds_list, analysis_dirs=analysis_dirs_matched, methods=INTER_DS_METHODS,
        outdir=sp_out,
        out_binem_outfile=f"{sp_out}/out_binem_similarity.png",
        cross_assay_binem_outfile=(f"{sp_out}/cross_assay_binem_similarity.png"
                                   if (CHIP_DATASETS and MINT_DATASETS) else None),
        group_a=CHIP_DATASETS, group_b=MINT_DATASETS))


### Optimal Number of States Analysis (Plotting)


In [ ]:
# Optimal Number of States Analysis (Plotting)
# Plotting results
if n_states_results:
    # Combined plots (mean with error across datasets per method)
    combined_rows = []
    _m_map = {"chromhmm": "ChromHMM", "homer": "HOMER", "macs2": "MACS2", "omni": "Omnipeak"}
    for (ds, m), res in n_states_results.items():
        for i, n in enumerate(n_states_range):
            combined_rows.append({
                "method": _m_map.get(m, m),
                "n_states": n,
                "inertia": res["inertia"][i],
                "silhouette": res["silhouette"][i],
                "entropy": res["entropy"][i]
            })
    df_plot = pd.DataFrame(combined_rows)

    # Combined Elbow
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_plot[~df_plot['inertia'].isna()], x='n_states', y='inertia', hue='method', marker='o', ax=ax)
    ax.set_xlabel('Number of States')
    ax.set_ylabel('Normalized Inertia')
    ax.set_title('Elbow Method (Normalized Inertia) - Combined')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_elbow.png")
    plt.close(fig)

    # Combined Silhouette
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_plot[~df_plot['silhouette'].isna()], x='n_states', y='silhouette', hue='method', marker='s', ax=ax)
    ax.set_xlabel('Number of States')
    ax.set_ylabel('Silhouette Score')
    ax.set_title('Silhouette Score - Combined')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_silhouette.png")
    plt.close(fig)

    # Combined Entropy
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_plot[~df_plot['entropy'].isna()], x='n_states', y='entropy', hue='method', marker='^', ax=ax)
    ax.set_xlabel('Number of States')
    ax.set_ylabel('Transition Matrix Entropy (bits)')
    ax.set_title('Transition Matrix Entropy - Combined')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_entropy.png")
    plt.close(fig)

    methods = sorted(set(m for d, m in n_states_results.keys()))
    for method in methods:
        method_label = "omnipeak" if method == "omni" else method
        # Plotting results - Elbow Method
        fig1, ax1 = plt.subplots(figsize=(10, 6))
        plotted_elbow = False
        for (ds, m), res in n_states_results.items():
            if m != method: continue
            if np.isnan(res["inertia"]).all(): continue
            label = f"{DS_TITLE.get(ds, ds)}"
            ax1.plot(list(n_states_range), res["inertia"], marker='o', label=label)
            plotted_elbow = True

        if plotted_elbow:
            ax1.set_xlabel('Number of States')
            ax1.set_ylabel('Normalized Inertia')
            ax1.set_title(f'Elbow Method (Normalized Inertia) - {method_label}')
            ax1.grid(True, linestyle='--', alpha=0.5)
            ax1.legend(fontsize='small', ncol=2)
            plt.tight_layout()
            os.makedirs("out", exist_ok=True)
            plt.savefig(f"out/optimal_n_states_elbow_{method_label}.png")
        plt.close(fig1)

        # Plotting results - Silhouette Score
        fig2, ax2 = plt.subplots(figsize=(10, 6))
        plotted_silhouette = False
        for (ds, m), res in n_states_results.items():
            if m != method: continue
            if np.isnan(res["silhouette"]).all(): continue
            label = f"{DS_TITLE.get(ds, ds)}"
            ax2.plot(list(n_states_range), res["silhouette"], marker='s', label=label)
            plotted_silhouette = True

        if plotted_silhouette:
            ax2.set_xlabel('Number of States')
            ax2.set_ylabel('Silhouette Score')
            ax2.set_title(f'Silhouette Score - {method_label}')
            ax2.grid(True, linestyle='--', alpha=0.5)
            ax2.legend(fontsize='small', ncol=2)
            plt.tight_layout()
            plt.savefig(f"out/optimal_n_states_silhouette_{method_label}.png")
        plt.close(fig2)

        # Plotting results - Transition Matrix Entropy
        fig3, ax3 = plt.subplots(figsize=(10, 6))
        plotted_entropy = False
        for (ds, m), res in n_states_results.items():
            if m != method: continue
            if "entropy" not in res or np.isnan(res["entropy"]).all(): continue
            label = f"{DS_TITLE.get(ds, ds)}"
            ax3.plot(list(n_states_range), res["entropy"], marker='^', label=label)
            plotted_entropy = True

        if plotted_entropy:
            ax3.set_xlabel('Number of States')
            ax3.set_ylabel('Transition Matrix Entropy (bits)')
            ax3.set_title(f'Transition Matrix Entropy - {method_label}')
            ax3.grid(True, linestyle='--', alpha=0.5)
            ax3.legend(fontsize='small', ncol=2)
            plt.tight_layout()
            plt.savefig(f"out/optimal_n_states_entropy_{method_label}.png")
        plt.close(fig3)


# 3. Showing the results

# Results
 Every plot produced by the computation cells above, displayed inline and
organised into five sections:

1. **Peaks — number and lengths**
2. **Segmentation — number of states and segments**
3. **Segmentation — state lengths**
4. **Segmentation — state composition**
5. **All other analyses** — entropy, similarity, biological validation,
   emission discriminability, cross-assay portability, replicate consistency
   and per-dataset detail.

Run the helper cell first; missing files are skipped silently, so the output
reflects exactly the segmentations that were produced.

In [ ]:
# ---------------------------------------------------------------------------
# Display helpers. Every plot below was produced by the computation cells above
# and is read straight off disk. Missing files are skipped silently, so the
# notebook renders cleanly regardless of which segmentations were produced.
# ---------------------------------------------------------------------------


def header(text, level=3):
    display(HTML(f"<h{level} style='border-bottom:1px solid #999;margin-top:1em'>{text}</h{level}>"))


def show(path, width=820, caption=None):
    """Display one image if it exists; return True when shown."""
    if not os.path.exists(path):
        return False
    if caption:
        display(HTML(f"<div style='color:#555;font-size:0.9em'>{caption}</div>"))
    display(Image(filename=path, width=width))
    return True


def show_group(title, items, width=820, level=3):
    """Show a titled group of plots; the title is skipped when nothing exists.

    Each item is either a path or a (path, caption) tuple.
    """
    norm = [(p, None) if isinstance(p, str) else p for p in items]
    present = [(p, c) for p, c in norm if os.path.exists(p)]
    if not present:
        return False
    header(title, level)
    for p, c in present:
        show(p, width=width, caption=c)
    return True


def show_table(path, caption=None):
    """Display a TSV as a DataFrame if it exists; return True when shown."""
    if not os.path.exists(path):
        return False
    if caption:
        display(HTML(f"<div style='color:#555;font-size:0.9em'>{caption}</div>"))
    display(pd.read_csv(path, sep="\t"))
    return True


def method_plot(ds, method_key, rel):
    """Path to a per-dataset, per-method analysis plot.

    The reference lives outside the variant dir (out/<ds>/ref/...).
    """
    if method_key == "ref":
        return f"out/{ds}/ref/{rel}"
    return f"out/{ds}/{VARIANT}/{method_key}/{rel}"


print("Display helpers ready (variant:", VARIANT + ").")


## 1. Peaks — number and lengths

Binarization peak statistics: peak count, mean peak length and gap lengths
between adjacent binarized elements, summarised across datasets and shown per
dataset (including replicate Jaccard where replicates exist).

In [ ]:
# 1. Peaks — number and lengths
for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {_gname}", [
        (f"{SP}/{_g}/peak_count.png", "Peak count per mark and method — mean \u00b1 std across datasets"),
        (f"{SP}/{_g}/peak_length.png", "Mean peak length per mark and method — mean \u00b1 std across datasets"),
        (f"{SP}/{_g}/peak_gap_violin.png", "Gap lengths between adjacent binarized elements (pooled across datasets)"),
    ], level=2)


## 2. Segmentation — number of states and segments

How fragmented each segmentation is: the total segment count per method across
datasets, the per-reference state/segment counts, and per-dataset
state/segment counts for every segmentation.

In [ ]:
# 2. Segmentation — number of states and segments
for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {_gname}", [
        (f"{SP}/{_g}/summary_n_segments.png",
         "Total number of segments per method (incl. ENCODE reference) — mean \u00b1 std across datasets"),
    ], level=2)

for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"ENCODE reference segmentations — {_gname}", [
        (f"{SP}/{_g}/reference_n_segments.png", "Number of segments per dataset's ENCODE reference"),
    ])


## 3. Segmentation — state lengths

Segment length distributions: the cross-dataset per-state comparison and
coverage, the ENCODE reference length summaries, and per-dataset segment-length
statistics with per-method length distributions.

In [ ]:
# 3. Segmentation — state lengths
show_group("Cross-dataset summary", [
    (f"{SP}/state_length_comparison.png", "Per-state segment length: reference vs all de-novo methods"),
    (f"{SP}/state_coverage.png", "Genomic coverage fraction per chromatin state"),
    (f"{SP}/summary_mean_tx_length.png", "Mean Tx (transcription) segment length"),
], level=2)

show_group("ENCODE reference segmentations", [
    (f"{REF}/mean_length.png", "Mean segment length"),
    (f"{REF}/median_length.png", "Median segment length"),
    (f"{REF}/min_length.png", "Min segment length"),
    (f"{REF}/max_length.png", "Max segment length"),
], width=750)


## 4. Segmentation — state composition

Fraction of the genome covered by each chromatin state: averaged per method
across datasets, across the ENCODE reference segmentations, and per dataset for
each de-novo method.

In [ ]:
# 4. Segmentation — state composition
show_group("Per-method composition (mean across datasets)", [
    (f"{SP}/method_state_composition.png", "State composition per method — mean across datasets"),
], level=2)

show_group("ENCODE reference composition", [
    (f"{REF}/state_composition.png", "State composition across ENCODE reference segmentations"),
])

comp = [(p, os.path.basename(p).replace("method_ds_composition_", "").replace(".png", ""))
        for p in sorted(glob.glob(f"{SP}/method_ds_composition_*.png"))]
show_group("Per-dataset composition for each de-novo method", comp)

comp2 = [(p, os.path.basename(p).replace("ds_composition_", "").replace(".png", ""))
         for p in sorted(glob.glob(f"{SP}/ds_composition_*.png"))]
show_group("Per-method composition for each dataset", comp2)


## 5. All other analyses

Transition entropy, inter-dataset / inter-reference similarity, biological
validation (RNA-seq / ATAC-seq), emission discriminability, cross-assay
portability, replicate consistency, the aggregated comparison table, and the
per-dataset detail (method comparison tables, functional enrichment and state
emissions for each method).

In [ ]:
# 5. All other analyses
show_group("Transition matrix entropy", [
    (f"{SP}/summary_entropy.png", "De-novo — Full (raw labels)"),
    (f"{SP}/summary_entropy_noqh.png", "De-novo — NOQH (excl. Quies/Het)"),
    (f"{REF}/entropy_summary_combined.png", "ENCODE reference entropy (Full + NOQH)"),
], level=2)

show_group("Segmentation similarity", [
    (f"{SP}/method_similarity_distribution.png", "De-novo inter-dataset similarity (3 variants) — Full"),
    (f"{SP}/method_similarity_distribution_noqh.png", "De-novo inter-dataset similarity (3 variants) — NOQH"),
    (f"{SP}/rep_consistency_distribution.png", "De-novo replicate consistency (3 variants) — Full"),
    (f"{SP}/rep_consistency_distribution_noqh.png", "De-novo replicate consistency (3 variants) — NOQH"),
    (f"{REF}/similarity_distribution.png", "Inter-reference similarity (3 variants) — Full"),
    (f"{REF}/similarity_distribution_noqh.png", "Inter-reference similarity (3 variants) — NOQH"),
    (f"{SP}/out_binem_similarity.png", "Inter-dataset binarized emission cosine similarity"),
    (f"{SP}/summary_kappa_vs_ref.png", "Kappa agreement vs ENCODE reference"),
    (f"{SP}/summary_jaccard_vs_ref.png", "Jaccard agreement vs ENCODE reference"),
], level=2)

show_group("Biological validation (RNA-seq / ATAC-seq)", [
    (f"{SP}/summary_jaccard_tx.png", "Jaccard: Tx state vs expressed gene bodies"),
    (f"{SP}/summary_enrich_tx.png", "Tx fold enrichment at expressed gene bodies"),
    (f"{SP}/summary_jaccard_tss.png", "Jaccard: Tss state vs RefSeq TSS ±1 kb"),
    (f"{SP}/summary_jaccard_tss_atac.png", "Jaccard: Tss state vs ATAC-seq peaks"),
], level=2)

show_group("Emission discriminability (Gini index)", [
    (f"{SP}/emission_gini_summary.png", "Gini index of state emissions"),
], level=2)

show_group("Cross-assay portability (ChIP ↔ Mint-ChIP)", [
    (f"{SP}/method_similarity_distribution_chip_vs_mint.png", "Full"),
    (f"{SP}/method_similarity_distribution_chip_vs_mint_noqh.png", "NOQH"),
    (f"{SP}/cross_assay_binem_similarity.png", "Cross-assay binarized emission similarity"),
], level=2)

show_group("Replicate consistency", sorted(glob.glob(f"{SP}/rep_consistency_*.png")), level=2)

header("Cross-dataset comparison table", 2)
if not show_table("out/comparison_table.tsv"):
    print("  (no aggregated comparison table)")



## 6. Per-state matching matrices

Work-state → ENCODE-reference matching score matrices produced by `match.py`. Rows are the de-novo method's states, columns the reference states; the Hungarian-selected match per row is outlined in red. Produced by the pipeline as `{...}_matched.match.png` next to each matched BED.

In [ ]:
# Uncomment to show
# # Per-state matching matrices (work → ENCODE reference).
# header("Per-state matching matrices (work → ENCODE reference)", 2)
# _match_methods = [("chromhmm_default", "Default ChromHMM"),
#                   ("kmeans_omni", "KMeans OmniPeak"),
#                   ("kmeans_homer", "KMeans HOMER"),
#                   ("kmeans_macs2", "KMeans MACS2")]
# for ds in DATASETS:
#     items = [(inter_ds_bed(ds, k).replace(".bed", ".match.png"),
#               f"{lbl}: per-state matching score (matched cell outlined in red)")
#              for k, lbl in _match_methods]
#     show_group(DS_TITLE.get(ds, ds), items, width=620, level=3)
#     break  # Comment to plot all the dataset plots


# 7. All other per dataset plots

In [ ]:
# Uncomment to show
# for ds in DATASETS:
#     show_group(DS_TITLE.get(ds, ds), [
#         (f"{ds}/peaks/n_peaks.png", "Number of peaks per mark"),
#         (f"{ds}/peaks/mean_length.png", "Mean peak length per mark"),
#         (f"{ds}/peaks/median_length.png", "Median peak length per mark"),
#         (f"{ds}/peaks/jaccard_rep1_vs_rep2.png", "Peak Jaccard: rep1 vs rep2"),
#     ], width=600, level=3)
#
#     show_group(DS_TITLE.get(ds, ds), [
#         (f"out/{ds}/{VARIANT}/n_segments.png", "Number of segments per segmentation"),
#         (f"out/{ds}/{VARIANT}/mean_Tx_length.png", "Mean Tx length per segmentation"),
#         (f"{SP}/per_state_kappa_{ds}.png", "Per-state Cohen's Kappa vs ENCODE reference"),
#     ], width=750, level=3)
#
#     show_group(DS_TITLE.get(ds, ds), [
#         (f"out/{ds}/{VARIANT}/mean_length.png", "Mean segment length per segmentation"),
#         (f"out/{ds}/{VARIANT}/median_length.png", "Median segment length per segmentation"),
#         (f"out/{ds}/{VARIANT}/min_length.png", "Min segment length per segmentation"),
#         (f"out/{ds}/{VARIANT}/max_length.png", "Max segment length per segmentation"),
#     ], width=750, level=3)
#     show_group(f"{DS_TITLE.get(ds, ds)} — per-method length distribution",
#                [(method_plot(ds, k, "segment_length.png"), lbl) for k, lbl in METHOD_LABELS],
#                width=600, level=4)
#
#     header(DS_TITLE.get(ds, ds), 3)
#     show_table(f"out/{ds}/{VARIANT}/comparison_table.tsv", caption="Method comparison table")
#     for k, lbl in METHOD_LABELS:
#         show_group(lbl, [
#             (method_plot(ds, k, "enrichment/enrichment.png"), f"{lbl}: functional enrichment"),
#             (method_plot(ds, k, "bin_emissions/state_emissions.png"), f"{lbl}: binarized emissions"),
#             (method_plot(ds, k, "bw_emissions/state_emissions.png"), f"{lbl}: bigwig emissions"),
#         ], width=760, level=4)

## 8. Optimal Number of States Results

In [ ]:
if n_states_results:
    show_group("Optimal Number of States Analysis - Combined", [
        ("out/optimal_n_states_elbow.png", "Elbow Method Analysis (Mean ± SE)"),
        ("out/optimal_n_states_silhouette.png", "Silhouette Score Analysis (Mean ± SE)"),
        ("out/optimal_n_states_entropy.png", "Transition Matrix Entropy Analysis (Mean ± SE)")
    ], width=800, level=3)

    methods = sorted(set(m for d, m in n_states_results.keys()))
    for method in methods:
        method_label = "omnipeak" if method == "omni" else method
        header(f"Method: {method_label}", level=3)
        show_group(f"Optimal Number of States Analysis - {method_label}", [
            (f"out/optimal_n_states_elbow_{method_label}.png", f"Elbow Method Analysis - {method_label}"),
            (f"out/optimal_n_states_silhouette_{method_label}.png", f"Silhouette Score Analysis - {method_label}"),
            (f"out/optimal_n_states_entropy_{method_label}.png", f"Transition Matrix Entropy Analysis - {method_label}")
        ], width=800, level=4)


# Cell type differences in chromatin

### 1. Compute

In [ ]:
os.makedirs("out/consistency", exist_ok=True)

# 1. Load segmentations
method_segs = defaultdict(list)
# We want to compare across all datasets for each method
for ds in CHIP_DATASETS:
    # Reference
    try:
        ref_path = ref_bed_path(ds)
        if os.path.exists(ref_path):
            method_segs["Reference"].append(match.load_bed(ref_path))
    except Exception as e:
        print(f"Warning: could not load reference for {ds}: {e}")
    
    # De-novo methods
    _consistency_methods = [("chromhmm_default", "ChromHMM"), 
                            ("kmeans_homer", "HOMER"), 
                            ("kmeans_macs2", "MACS2"), 
                            ("kmeans_omni", "Omnipeak")]
    for k, lbl in _consistency_methods:
        try:
            path = inter_ds_bed(ds, k)
            if os.path.exists(path):
                method_segs[lbl].append(match.load_bed(path))
        except Exception as e:
            print(f"Warning: could not load {lbl} for {ds}: {e}")

# 2. Compute consistency
method_counts = {}
for method_name, segs_list in method_segs.items():
    if not segs_list: continue
    cache_path = f"out/consistency/{method_name.lower()}.pkl"
    if os.path.exists(cache_path):
        print(f"Loading cached consistency for {method_name}...")
        with open(cache_path, "rb") as f:
            method_counts[method_name] = pickle.load(f)
    else:
        print(f"--- Analyzing {method_name} ({len(segs_list)} datasets) ---")
        counts = analyze.compute_state_consistency(segs_list, bin_size=CHROMHMM_BIN, show_progress=True)
        with open(cache_path, "wb") as f:
            pickle.dump(counts, f)
        method_counts[method_name] = counts


### 2. Plotting

In [ ]:
# Use colors from the first reference dataset if possible
ref_colors = {}
if CHIP_DATASETS:
    for ds in CHIP_DATASETS:
        rp = ref_bed_path(ds)
        if os.path.exists(rp):
            ref_colors = match.state_colors(match.load_bed(rp))
            break

for method_name, counts in method_counts.items():
    print(f"--- Plotting {method_name} ---")
    out_path = f"out/consistency/{method_name.lower()}_consistency.png"
    analyze.plot_state_consistency(counts, method_name, out_path, colors=ref_colors)


### 3. Display

In [ ]:
for method_name in ["Reference", "ChromHMM", "HOMER", "MACS2", "Omnipeak"]:
    img_path = f"out/consistency/{method_name.lower()}_consistency.png"
    show(img_path, caption=method_name)
